# Mapa sistemático da literatura — versão sem programação

Este caderno executa o protocolo de recuperação e curadoria de literatura
([repositório](https://github.com/ProfJoaoGuilherme/Mapa-sistematico-py)) a partir de uma **planilha** que descreve o seu
tema. Nada precisa ser instalado: tudo roda no Google Colab, no navegador.

**Como usar:** clique em cada célula, na ordem, e aperte o botão ▶ à esquerda
dela (ou `Ctrl+Enter`). Espere terminar uma antes de rodar a próxima. Os campos
à direita das células são os únicos lugares onde você escreve.

| Etapa | O que acontece | Tempo típico |
|---|---|---|
| 1. Preparar | baixa os programas e cria a pasta do seu tema | 1 min |
| 2. Chaves | e-mail e chave do CORE, opcionais | — |
| 3. Planilha | carrega a sua planilha | 1 min |
| 4. Conferir | traduz a planilha e mostra a equação de busca | segundos |
| 5. Estimar | conta quantos registros o OpenAlex devolveria | segundos |
| 6. Buscar | consulta as bases | minutos a horas |
| 7–9. Triar, consolidar, ordenar | elegibilidade, repetições, lista final | minutos |
| 10–11. Ver e baixar | mostra a lista e entrega os arquivos | 1 min |

O guia completo, com o preenchimento da planilha passo a passo, está em
[`GUIA_SEM_PROGRAMACAO.md`](https://github.com/ProfJoaoGuilherme/Mapa-sistematico-py/blob/main/GUIA_SEM_PROGRAMACAO.md).

## 1. Preparar o ambiente

Marque **salvar_no_drive** para guardar a pasta do tema no seu Google Drive. É
o recomendado: se o Colab desconectar no meio da busca, os arquivos já gravados
continuam lá. O Colab vai pedir permissão de acesso ao Drive.

In [ ]:
#@title 1. Preparar o ambiente { display-mode: "form" }
salvar_no_drive = True  #@param {type:"boolean"}
nome_da_pasta = "protocolo_meu_tema"  #@param {type:"string"}

import glob, os, re, shutil, subprocess, sys

REPO = "https://github.com/ProfJoaoGuilherme/Mapa-sistematico-py.git"
try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB and salvar_no_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    raiz = "/content/drive/MyDrive"
else:
    raiz = "/content" if NO_COLAB else os.getcwd()

PASTA = os.path.join(raiz, nome_da_pasta.strip() or "protocolo_meu_tema")
CODIGO = os.environ.get("PROTOCOLO_CODIGO") or "/content/protocolo_codigo"
if not os.path.exists(CODIGO):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODIGO], check=True)

os.makedirs(PASTA, exist_ok=True)
for arquivo in ["buscar.py", "triar_e_codificar.py", "consolidar_vias.py",
                "ranquear.py", "relatorio_funil.py", "planilha_para_config.py",
                "visualizador.html", "modelo_configuracao.xlsx",
                "modelo_configuracao_em_branco.xlsx"]:
    shutil.copy(os.path.join(CODIGO, arquivo), PASTA)
os.chdir(PASTA)
if PASTA not in sys.path:
    sys.path.insert(0, PASTA)


def rodar(*comando):
    """Executa um programa do protocolo e mostra a saída enquanto ele roda."""
    p = subprocess.Popen([sys.executable, *comando], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linha in p.stdout:
        print(linha, end="")
    if p.wait() != 0:
        raise RuntimeError("A etapa falhou. Leia a mensagem acima, corrija e "
                           "rode a célula de novo.")


print("Pronto. Pasta do tema:", PASTA)

## 2. Chaves de acesso (opcional)

- **email**: o OpenAlex atende mais rápido quem se identifica. Não é publicado.
- **chave_core**: obrigatória só se a base `core` estiver na planilha. É
  gratuita: cadastre-se em <https://core.ac.uk/services/api>.

Deixe em branco o que não usar.

In [ ]:
#@title 2. Chaves de acesso { display-mode: "form" }
email = ""  #@param {type:"string"}
chave_core = ""  #@param {type:"string"}

if email.strip():
    os.environ["OPENALEX_MAILTO"] = email.strip()
    os.environ["CROSSREF_MAILTO"] = email.strip()
if chave_core.strip():
    os.environ["CORE_API_KEY"] = chave_core.strip()
print("e-mail:", "informado" if email.strip() else "não informado",
      "| chave do CORE:", "informada" if chave_core.strip() else "não informada")

## 3. Carregar a planilha

Três maneiras, escolha uma:

1. **Planilha Google**: cole o link em `link_da_planilha`. Antes, no Google
   Planilhas, clique em *Compartilhar* → *Qualquer pessoa com o link* → *Leitor*.
2. **Arquivo do computador**: deixe o link vazio e rode a célula. Aparece um
   botão *Escolher arquivos*; selecione o seu `.xlsx`.
3. **Exemplo**: marque `usar_modelo_de_exemplo` para rodar com a planilha do
   estudo de referência (áreas úmidas) e ver como tudo funciona.

In [ ]:
#@title 3. Carregar a planilha { display-mode: "form" }
link_da_planilha = ""  #@param {type:"string"}
usar_modelo_de_exemplo = False  #@param {type:"boolean"}

import urllib.request

PLANILHA = "configuracao.xlsx"
if usar_modelo_de_exemplo:
    shutil.copy("modelo_configuracao.xlsx", PLANILHA)
elif link_da_planilha.strip():
    m = re.search(r"/d/([\w-]+)", link_da_planilha)
    if not m:
        raise RuntimeError("Link não reconhecido. Copie o endereço da planilha "
                           "como aparece no navegador (contém /d/...).")
    url = ("https://docs.google.com/spreadsheets/d/%s/export?format=xlsx"
           % m.group(1))
    with urllib.request.urlopen(url) as r:
        dados = r.read()
    if not dados.startswith(b"PK"):
        raise RuntimeError("O Google não entregou a planilha. Confira se ela está "
                           "compartilhada como 'Qualquer pessoa com o link'.")
    with open(PLANILHA, "wb") as fh:
        fh.write(dados)
elif NO_COLAB:
    from google.colab import files
    enviados = files.upload()
    if not enviados:
        raise RuntimeError("Nenhum arquivo enviado.")
    nome = next(iter(enviados))
    if nome != PLANILHA:
        shutil.move(nome, PLANILHA)
elif not os.path.exists(PLANILHA):
    raise RuntimeError("Coloque a planilha nesta pasta com o nome configuracao.xlsx.")
print("Planilha carregada:", os.path.join(PASTA, PLANILHA))

## 4. Conferir a configuração

O programa lê a planilha, aponta os problemas encontrados e, se não houver
nenhum, mostra a equação de busca que será enviada às bases. **Leia com
atenção.** Se algo estiver errado, corrija a planilha e volte à etapa 3.

In [ ]:
#@title 4. Conferir a configuração { display-mode: "form" }
rodar("planilha_para_config.py", PLANILHA)

## 5. Estimar o volume (recomendado)

Conta, sem baixar nada, quantos registros o OpenAlex tem para a equação de
cada idioma. Serve para calibrar os termos antes da busca demorada.

- **Dezenas de milhares**: a equação está ampla demais. Restrinja o bloco do
  tema ou o da fonte.
- **Poucas dezenas**: ampla de menos. Acrescente sinônimos.
- O número maior que `limite_por_consulta` (aba *geral*) significa que só os
  registros mais relevantes, até esse teto, serão baixados.

In [ ]:
#@title 5. Estimar o volume no OpenAlex { display-mode: "form" }
import importlib, json, urllib.parse, urllib.request
import config
importlib.reload(config)

a0, a1 = config.PERIODO
for idioma, b in config.BLOCOS.items():
    eq = " AND ".join("(" + " OR ".join(b[k]) + ")"
                      for k in ("tema", "fonte", "abordagem"))
    url = "https://api.openalex.org/works?" + urllib.parse.urlencode({
        "search": eq, "per-page": "1",
        "filter": f"from_publication_date:{a0}-01-01,to_publication_date:{a1}-12-31"})
    with urllib.request.urlopen(url, timeout=60) as r:
        n = json.load(r)["meta"]["count"]
    aviso = "  (acima do teto: serão baixados %d)" % config.MAX_POR_CONSULTA \
        if n > config.MAX_POR_CONSULTA else ""
    print("[%s] %d registros%s" % (idioma, n, aviso))

## 6. Buscar nas bases

É a etapa demorada. O OpenAlex costuma levar minutos; a OASISBR pode levar
horas, porque consulta cada par de termos separadamente. **Mantenha a aba do
navegador aberta.** O Colab gratuito desconecta após cerca de 90 minutos sem
interação ou 12 horas de uso.

Para dividir o trabalho, escreva em `bases_desta_vez` só uma base (por exemplo
`openalex`) e rode de novo depois com a outra (`oasisbr`). Cada base grava o
seu próprio arquivo `bruto_<base>.csv` e nada se perde entre rodadas. Deixe
vazio para buscar em todas as bases da planilha.

In [ ]:
#@title 6. Buscar { display-mode: "form" }
bases_desta_vez = ""  #@param {type:"string"}

escolhidas = [b for b in re.split(r"[,;\s]+", bases_desta_vez.strip()) if b]
rodar("buscar.py", *escolhidas)
print("\narquivos de busca nesta pasta:", sorted(glob.glob("bruto_*.csv")))

## 7. Triagem e codificação

Aplica os critérios da planilha a cada registro. Para cada base saem três
arquivos: `incluidos_`, `excluidos_` (com o motivo) e `triagem_manual_`, este
último ordenado por evidência para conferência humana.

Olhe os **motivos de exclusão**: se quase tudo cai num mesmo critério, ele
pode estar estreito demais.

In [ ]:
#@title 7. Triar e codificar { display-mode: "form" }
brutos = sorted(glob.glob("bruto_*.csv"))
if not brutos:
    raise RuntimeError("Nenhum bruto_*.csv. Rode a etapa 6 antes.")
for bruto in brutos:
    print("=" * 70, "\n", bruto)
    rodar("triar_e_codificar.py", bruto)

## 8. Consolidar as bases

Reúne as bases e remove os trabalhos repetidos entre elas, pelo DOI e, na
falta dele, pelo título. Quando um trabalho aparece em duas bases, fica
creditado à primeira desta ordem: OpenAlex, OASISBR, CORE, BDTD, Crossref,
Semantic Scholar.

In [ ]:
#@title 8. Consolidar { display-mode: "form" }
ORDEM = ["openalex", "oasisbr", "core", "bdtd", "crossref", "semanticscholar"]
incluidos = sorted(glob.glob("incluidos_*.csv"),
                   key=lambda f: (ORDEM.index(f[10:-4]) if f[10:-4] in ORDEM
                                  else len(ORDEM), f))
if not incluidos:
    raise RuntimeError("Nenhum trabalho passou na triagem. Revise os critérios "
                       "da planilha (etapa 7 mostra os motivos).")
rodar("consolidar_vias.py", *incluidos)

## 9. Ordenar e relatar

Gera a **lista de leitura ordenada por relevância** e o relatório do funil
(quantos registros entraram e saíram em cada etapa, e quantos trabalhos
mencionam cada atributo).

In [ ]:
#@title 9. Ordenar e gerar o relatório { display-mode: "form" }
rodar("ranquear.py", "corpus_consolidado.csv")
rodar("relatorio_funil.py")
print()
print(open("relatorio_funil.txt", encoding="utf-8").read())

## 10. Ver o início da lista

A lista não corta nenhum trabalho: ela ordena, e **a leitura decide**. A coluna
`cobertura_conceitos` diz quantos dos conceitos da planilha cada trabalho
cobre.

In [ ]:
#@title 10. Ver os primeiros trabalhos { display-mode: "form" }
quantos = 25  #@param {type:"integer"}
import pandas as pd

lista = pd.read_csv("lista_leitura_priorizada.csv", dtype=str)
colunas = [c for c in ["ordem", "cobertura_conceitos", "ano", "autor", "titulo",
                       "periodico", "via", "doi"] if c in lista.columns]
pd.set_option("display.max_colwidth", 90)
lista[colunas].head(quantos)

## 11. Baixar os resultados

Gera `resultados.zip` com a lista em planilha (`.xlsx`) e em `.csv`, o
relatório do funil, as planilhas de triagem manual, a configuração usada e o
registro das consultas. Se você salvou no Drive, os arquivos também estão lá.

In [ ]:
#@title 11. Baixar os resultados { display-mode: "form" }
import zipfile
import pandas as pd

pd.read_csv("lista_leitura_priorizada.csv", dtype=str).to_excel(
    "lista_leitura_priorizada.xlsx", index=False)
entregar = (["lista_leitura_priorizada.xlsx", "lista_leitura_priorizada.csv",
             "relatorio_funil.txt", "corpus_consolidado.csv", "config.py",
             PLANILHA, "log_busca.csv", "visualizador.html"]
            + sorted(glob.glob("triagem_manual_*.csv")))
with zipfile.ZipFile("resultados.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for arquivo in entregar:
        if os.path.exists(arquivo):
            z.write(arquivo)
print("gerado:", os.path.join(PASTA, "resultados.zip"))
if NO_COLAB:
    from google.colab import files
    files.download("resultados.zip")

## Ler a lista com o visualizador

Descompacte `resultados.zip`, abra `visualizador.html` no navegador do seu
computador e carregue `lista_leitura_priorizada.csv`. Ele oferece busca por
texto, filtros, exportação em RIS para o Zotero e um campo para registrar a
decisão de inclusão de cada trabalho.

## Se algo der errado

| Sintoma | O que fazer |
|---|---|
| A etapa 4 lista problemas | Corrija as linhas indicadas na planilha e repita as etapas 3 e 4. |
| `HTTP 429` ou `erro 5xx` na busca | A base está sobrecarregada. Espere alguns minutos e rode a etapa 6 só com essa base. |
| O Colab desconectou | Rode a etapa 1 de novo (mesmo nome de pasta) e continue de onde parou. Com o Drive, os `bruto_*.csv` já gravados continuam lá. |
| Nenhum trabalho incluído | Os critérios estão estreitos demais. Veja os motivos de exclusão na etapa 7. |
| Lista com muito ruído | Acrescente termos na aba *exclusao* ou marque `palavra_inteira` nas siglas curtas. |